# Lab 36: Datasets, Features, Labels and Data Preparation

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
Most of the work in machine learning is not the model. It is the **data**. A model learns whatever the data teaches it, including the mistakes. In this lab you take a messy table of sensor readings, the kind that real sensors produce, and make it ready for learning: you look at it, clean it, choose the **features** and the **label**, and split it in a way that does not cheat. You meet **data leakage**, the most common reason why a model looks great in the lab and fails in real life, and you build a preparation pipeline that gives the same result every time.

**By the end of this lab you can:**
1. Inspect a dataset: size, types, missing values, duplicates and impossible values.
2. Clean a table with pandas: text, duplicates, outliers and missing values.
3. Choose features and a label, and create a new feature from existing ones.
4. Recognise and avoid data leakage (a feature that hides the answer, scaling before the split, a bad split).
5. Build a reproducible preparation pipeline that is fitted on the training data only.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Classification of this lab.** Machine learning *preparation*, with pandas and scikit-learn. No neural network is trained in this lab.

## 2. Background

**Garbage in, garbage out.** A model cannot know that `999` degrees is a broken sensor. It will learn it. So before you train, you check the data.

**The preparation steps.**

| Step | Question | Tool |
|---|---|---|
| Inspect | What is in the table? How big? What types? | `head`, `info`, `describe` |
| Clean | Typos, duplicates, impossible values, missing values | `str.strip`, `drop_duplicates`, a valid range, `fillna` |
| Choose | Which columns are features? Which is the label? | your decision |
| Create | Can a new column help (hour of the day)? | feature engineering |
| Split | Train, validation, test, without cheating | `train_test_split`, or by time |
| Scale | Put the numbers on the same scale | `StandardScaler` (fit on the train set only) |

**Missing values.** A sensor can fail and leave an empty cell (`NaN`). You can drop the row (you lose data), fill with the **median** of the column (the middle value; strong against outliers) or fill with a value that makes sense. The median must be calculated from the **training** set only.

**Data leakage.** *Leakage* means the model gets information during training that it will not have in real use. The test then looks excellent but means nothing. Common causes:

| Leak | Example | Fix |
|---|---|---|
| A feature that contains the answer | A column `fan_power_w` that is 0 when the fan is off | Remove it. Ask: *will I know this when I must decide?* |
| Preparing before splitting | Scaling or filling with numbers from all the rows | Split first, fit on the train set, apply to the others |
| The same example in train and test | Duplicate rows | Remove duplicates before the split |
| Mixing the future into the past | A random split of time-ordered data | Split by time: the test set is the newest data |

**Reproducible.** A pipeline is a list of steps that you can run again on new data and get the same treatment. In scikit-learn, `Pipeline` keeps the steps together, so that `fit` learns from the train set and `transform` only applies.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(36)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (10 min)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 7
rng = np.random.default_rng(SEED)
print("pandas", pd.__version__)

---
# Part B: A Messy Table (30 min)

### Step 1: The Data
The cell writes a file `readings.csv` with 600 readings of three rooms, as a careless logger would. It has typos in the room names, empty cells, repeated rows, a broken sensor (999) and a column `fan_power_w` that you will meet again. The column `fan` is the label: 1 when the fan was on. Open the file in the file browser if you like.

In [ ]:
N = 600
times = pd.date_range("2026-03-02 06:00", periods=N, freq="15min")
room = rng.choice(["kitchen", "lab", "garage"], N)
hour = times.hour.to_numpy() + times.minute.to_numpy() / 60
base = np.select([room == "kitchen", room == "lab"], [26.0, 24.0], default=28.0)
day = np.arange(N) / 96                                     # 96 readings per day: the weather gets warmer day by day
temperature = (base + 4 * np.sin((hour - 9) / 24 * 2 * np.pi) + 0.8 * day + rng.normal(0, 1.0, N)).round(1)
humidity = (60 - 0.8 * (temperature - 25) * 3 + rng.normal(0, 8, N)).round(0)
feels_hot = temperature + 0.08 * (humidity - 55) + rng.normal(0, 1.0, N)      # the fan is on when it feels too hot
fan = (feels_hot > 27.5).astype(int)
fan_power_w = np.where(fan == 1, rng.normal(32, 2, N), 0).round(1)

df = pd.DataFrame({"time": times.astype(str), "room": room, "temperature": temperature, "humidity": humidity, "fan_power_w": fan_power_w, "fan": fan})
df["room"] = [r.capitalize() if i % 7 == 0 else (" " + r if i % 11 == 0 else r) for i, r in enumerate(df.room)]     # typos
df.loc[rng.choice(N, 25, replace=False), "temperature"] = np.nan                                                    # empty cells
df.loc[rng.choice(N, 20, replace=False), "humidity"] = np.nan
df.loc[[50, 300, 420], "temperature"] = 999.0                                                                       # broken sensor
df = pd.concat([df, df.sample(30, random_state=1)]).reset_index(drop=True)                                          # repeated rows
df.to_csv("readings.csv", index=False)
print("Written readings.csv:", len(df), "rows")

### Step 2: Inspect Before You Touch
Read the table with pandas. `info` shows the types and the number of non-empty cells, `describe` shows the minimum and maximum, `value_counts` counts the names.

In [ ]:
raw = pd.read_csv("readings.csv")
print(raw.head())
print()
raw.info()
print()
print(raw.describe().round(1))
print()
print("Room names:", raw.room.value_counts().to_dict())
print("Empty cells per column:", raw.isna().sum().to_dict())
print("Repeated rows:", int(raw.duplicated().sum()))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many rows and columns:
- How many different room names are there, and how many should there be:
- What is the largest temperature, and what does it tell you:
- How many empty cells and repeated rows:

---
# Part C: Clean It (35 min)

### Step 1: Text, Duplicates and Impossible Values
Three jobs: clean the room names (`strip` removes spaces, `lower` makes small letters), drop the repeated rows, and turn impossible temperatures into empty cells (the sensor works between -10 and 50 degrees; anything outside is broken). **Do not fill the empty cells yet**: that needs the training set.

In [ ]:
clean = raw.copy()
clean["room"] = clean.room.str.strip().str.lower()
clean = clean.drop_duplicates().reset_index(drop=True)
broken = (clean.temperature < -10) | (clean.temperature > 50)
print("Broken temperatures:", int(broken.sum()))
clean.loc[broken, "temperature"] = np.nan

print("Rooms now:", clean.room.value_counts().to_dict())
print("Rows:", len(raw), "->", len(clean))
print("Temperature range:", clean.temperature.min(), "to", clean.temperature.max())
print("Empty cells:", clean[["temperature", "humidity"]].isna().sum().to_dict())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Rows before and after cleaning:
- How many broken temperatures were found:
- Why do we turn the broken values into empty cells and not delete the rows:

### Step 2: A Feature from the Time
The time is text now. A model cannot use `2026-03-02 06:15`, but it can use the **hour of the day**, because the room gets warm in the afternoon. This is **feature engineering**: making a useful column from the existing ones.

In [ ]:
clean["time"] = pd.to_datetime(clean.time)
clean["hour"] = clean.time.dt.hour + clean.time.dt.minute / 60
print(clean[["time", "room", "hour", "temperature", "humidity", "fan"]].head(5))

### Step 3: Look at It Again
The picture shows the temperature in each hour for the three rooms. Is the pattern what you expect?

In [ ]:
plt.figure(figsize=(6, 3.4))
for name, group in clean.groupby("room"):
    plt.scatter(group.hour, group.temperature, s=8, label=name)
plt.xlabel("hour of the day")
plt.ylabel("temperature (C)")
plt.legend()
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which room is the warmest and at what time of the day:
- What would the picture look like if we had not removed the 999 values:

---
# Part D: Features, Label and the Leak (35 min)

### Step 1: Test a Model with a Leaky Feature
A quick model shows the danger. The cell trains a decision tree to predict `fan` (1) with `fan_power_w` among the features, and (2) without it. The first is almost perfect, and useless: when you must decide if the fan should be on, you do not know yet how much power the fan uses.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

work = clean.dropna(subset=["temperature", "humidity"]).copy()
work["room_code"] = work.room.map({"kitchen": 0, "lab": 1, "garage": 2})
y = work.fan

for features in (["temperature", "humidity", "hour", "room_code", "fan_power_w"], ["temperature", "humidity", "hour", "room_code"]):
    X_a, X_b, y_a, y_b = train_test_split(work[features], y, test_size=0.25, random_state=SEED, stratify=y)
    tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_a, y_a)
    print(f"{'with' if 'fan_power_w' in features else 'without':<8} fan_power_w: test accuracy {tree.score(X_b, y_b) * 100:.1f} %   (the tree uses: {[f for f, v in zip(features, tree.feature_importances_) if v > 0]})")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test accuracy with and without fan_power_w:
- Which feature did the tree use when fan_power_w was there:
- Ask the question: will I know fan_power_w when I must decide? What is your answer, and what do you do with the column:

### Step 2: Another Leak, Scaling Before the Split
Fill and scale the whole table first and split afterwards, and the test rows have helped to calculate the median and the mean. The data is in time order and the weather gets warmer, so the numbers differ clearly. The cell shows how different they are.

In [ ]:
X = clean[["temperature", "humidity", "hour"]]
y = clean.fan
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)

by_time = clean.sort_values("time").reset_index(drop=True)
old_part, new_part = by_time[: int(len(by_time) * 0.75)], by_time[int(len(by_time) * 0.75):]
wrong_median = by_time.temperature.median()           # uses ALL rows, including the newest rows (the test rows)
right_median = old_part.temperature.median()          # uses the train rows only
print(f"Median temperature from all rows  : {wrong_median}   <- the future leaked in")
print(f"Median temperature from train rows: {right_median}")
print(f"Median temperature of the test rows: {new_part.temperature.median()}  (the weather got warmer)")
print("With a random split the two medians are almost equal. With data in time order they are not.")

---
# Part E: A Pipeline That Does Not Cheat (35 min)

### Step 1: Build the Pipeline
A `Pipeline` has two steps: `SimpleImputer(strategy="median")` fills the empty cells with the median of each column, and `StandardScaler` puts every column on the same scale. `fit` learns the medians, the means and the spreads from the **train set only**. `transform` applies them to any data.

In [ ]:
prep = Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
prep.fit(X_train)                                                  # learns from the train set only
train_ready = prep.transform(X_train)
test_ready = prep.transform(X_test)

print("Medians learned from the train set:", dict(zip(X.columns, prep.named_steps["fill"].statistics_.round(1))))
print("Means learned from the train set:  ", dict(zip(X.columns, prep.named_steps["scale"].mean_.round(1))))
print("Empty cells left:", int(np.isnan(train_ready).sum() + np.isnan(test_ready).sum()))
print("Train features after the pipeline: mean", train_ready.mean(axis=0).round(2), "spread", train_ready.std(axis=0).round(2))
print("Test features after the pipeline:  mean", test_ready.mean(axis=0).round(2), "(not exactly 0: that is correct)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The median temperature that the pipeline learned:
- Why is the mean of the test features not exactly 0:
- What happens if you call prep.fit on the test set:

### Step 2: A Split by Time
In real use the model predicts the **future**. A random split mixes the future into the training set. A split by time is honest: the oldest 75 % is the train set, the newest 25 % is the test set. Compare the scores of a tree with the two splits.

In [ ]:
ordered = clean.sort_values("time").reset_index(drop=True)
cut = int(len(ordered) * 0.75)
cols = ["temperature", "humidity", "hour"]
splits = {"random split": train_test_split(ordered[cols], ordered.fan, test_size=0.25, random_state=SEED, stratify=ordered.fan),
          "split by time": (ordered[cols][:cut], ordered[cols][cut:], ordered.fan[:cut], ordered.fan[cut:])}
for name, (a, b, ya, yb) in splits.items():
    model = Pipeline([("fill", SimpleImputer(strategy="median")), ("tree", DecisionTreeClassifier(max_depth=3, random_state=SEED))]).fit(a, ya)
    print(f"{name:<14} train {len(a)}  test {len(b)}  test accuracy {model.score(b, yb) * 100:.1f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The two accuracies (they can be close on this data):
- Which one would you trust for a system that predicts tomorrow, and why. If the two numbers are the same, why is the split by time still the right test:

### Step 3: Save the Prepared Data
The cell saves the prepared train and test sets and the pipeline. You use the files in Lab 37 and Lab 41.

In [ ]:
import joblib

pd.DataFrame(train_ready, columns=cols).assign(fan=y_train.to_numpy()).to_csv("train_ready.csv", index=False)
pd.DataFrame(test_ready, columns=cols).assign(fan=y_test.to_numpy()).to_csv("test_ready.csv", index=False)
joblib.dump(prep, "prep.joblib")
print("Saved train_ready.csv, test_ready.csv and prep.joblib")
print(pd.read_csv("train_ready.csv").head(3).round(2))

---
# Part F: Make It Your Own and Present (25 min)
Choose **one**:

- Add a feature that you think helps (for example *outside hot hours*: 1 when the hour is between 12 and 16) and check whether the tree gets better.
- Handle the missing values in another way (drop the rows, or fill with the mean) and compare the accuracy.
- Find a second leak: make a column that hides the answer and show that the accuracy jumps.
- Make the dirty data **worse** (more empty cells) and see at which point the model fails.

Then give a **3-minute demonstration**: show the messy table, the cleaning, the leak, and the pipeline.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- What you saw:
- One habit you will keep from this lab:

> **Where next.** In Lab 37 you train classical machine learning models (trees, nearest neighbours, regression) on prepared data and compare them fairly with a baseline.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The messy table is inspected: size, types, empty cells, repeated rows and the impossible value | |
| 2 | The cleaning is done and the row counts are explained | |
| 3 | The hour feature is created and the picture is read | |
| 4 | The leak with `fan_power_w` is shown and the column is removed for the right reason | |
| 5 | The pipeline is fitted on the train set only and the numbers it learned are read | |
| 6 | The random split and the split by time are compared | |
| 7 | The prepared files are saved | |
| 8 | One improvement is done and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. Why is the data more important than the model?:
- 2. Why do we turn a broken value into an empty cell?:
- 3. Why use the median to fill empty cells, and why from the train set only?:
- 4. What is data leakage? Give two examples from this lab.:
- 5. What is the question you ask to find a leaky feature?:
- 6. What does `fit` do in a pipeline and what does `transform` do?:
- 7. Why is a split by time more honest than a random split for sensor data?:
- 8. What is feature engineering? Give your own example.:
- 9. 30 rows are repeated in the table. How can this make the test score too good?:
- 10. A friend scaled all the data and then split it. What is wrong, and how large might the problem be?:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `FileNotFoundError: readings.csv` | Step 1 of Part B was not run | Run the cell that writes the file |
| `KeyError: 'room_code'` | The cell of Part D Step 1 was not run before | Run the cells from the top |
| `ValueError: Input X contains NaN` | The model was fitted without the imputer | Use the pipeline (it fills the empty cells) or drop the empty rows |
| The numbers are different from the example | The random seed or the order of the cells | Run from the top. The seed is 7. |
| The picture is empty | The temperature still has NaN or no rows | Run Part C again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(36)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- pandas, Getting started: https://pandas.pydata.org/docs/getting_started/index.html
- scikit-learn, Common pitfalls (data leakage): https://scikit-learn.org/stable/common_pitfalls.html
- scikit-learn, Pipelines: https://scikit-learn.org/stable/modules/compose.html